In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from tqdm import tqdm
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, HistoricAverage, AutoARIMA, Naive

In [ ]:
# Constants

KAGGLE_INPUT_BASE_PATH = "/kaggle/input/m5-forecasting-accuracy"
LOCAL_INPUT_BASE_PATH = "./data"
INPUT_BASE_PATH = LOCAL_INPUT_BASE_PATH

NIXTLA_RESULTS_PATH = Path("./results/nixtla/")
NIXTLA_RESULTS_PATH.mkdir(parents=True, exist_ok=True)

MAX_TRAIN_INDEX = 1941
FORECAST_HORIZON = 28
SUBMISSION_F_COLS = [f"F{i}" for i in range(1, FORECAST_HORIZON + 1)]

In [ ]:
# Load data

CALENDAR_DATA = pl.read_csv(f"{INPUT_BASE_PATH}/calendar.csv", try_parse_dates=True)
SALES_TRAIN_EVALUATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_evaluation.csv")
SALES_TRAIN_VALIDATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_validation.csv")
SAMPLE_SUBMISSION = pl.read_csv(f"{INPUT_BASE_PATH}/sample_submission.csv")

### Preprocess

In [ ]:
SALES_TRAIN_EVALUATION.head()

In [ ]:
train_df = (
    SALES_TRAIN_EVALUATION.unpivot(
        index=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
        variable_name="d",
        value_name="sales",
    )
    .join(
        CALENDAR_DATA.select(pl.col("date"), pl.col("d")),
        on="d",
        how="left",
    )
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int64))
    .select(pl.col("id"), pl.col("sales"), pl.col("d_index"))
)

train_df.head()

In [ ]:
# Plot random selection of time series
sample_ids = train_df["id"].sample(10).to_list()

StatsForecast.plot(
    df=train_df,
    unique_ids=sample_ids,
    id_col="id",
    target_col="sales",
    time_col="d_index",
)

In [ ]:
# Define models to train

models = [Naive(), HistoricAverage(), AutoARIMA(), AutoETS(season_length=7)]
model_names = [m.__class__.__name__ for m in models]

# Instantiate StatsForecast class as sf
sf = StatsForecast( 
    models=models,
    freq=1,
    fallback_model=Naive(),
    n_jobs=-1,
)

sf.fit(
    train_df.filter(pl.col("id").is_in(sample_ids)),
    id_col="id",
    target_col="sales",
    time_col="d_index",
)

In [ ]:
# Procude forecasts 

stats_forecast_df = sf.forecast(
    df=train_df.filter(pl.col("id").is_in(sample_ids)),
    h=FORECAST_HORIZON,
    id_col="id",
    target_col="sales",
    time_col="d_index",
)

stats_forecast_df.head()

In [ ]:
sf.plot(
    df=train_df.filter(pl.col("id").is_in(sample_ids)),
    forecasts_df=stats_forecast_df,
    # models=["AutoARIMA", "AutoETS"],
    # unique_ids=["FOODS_1_210_TX_1_evaluation"],
    # level=[90],                                   
    id_col="id",
    time_col="d_index",
    target_col="sales",
    max_insample_length=50,
)

In [ ]:
stats_forecast_df.head()

In [ ]:
# Save forecasts

for model_name in model_names:
    wide_model_forecast_df = (
        stats_forecast_df
        .select(pl.col("id"), pl.col("d_index"), pl.col(model_name).round().cast(pl.Int64))
        .with_columns(F_index=pl.format("F{}", pl.col("d_index") - MAX_TRAIN_INDEX))
        .pivot(index="id", on="F_index", values=model_name)
    )
    model_submission_df = (
        SAMPLE_SUBMISSION
        .join(wide_model_forecast_df, on="id", how="left", suffix=f"_{model_name}")
        .with_columns(
            pl.coalesce(pl.col(f"{c}_{model_name}"), pl.col(c)).cast(pl.Int64).alias(c)
            for c in SUBMISSION_F_COLS
        )
        .select(SAMPLE_SUBMISSION.columns)
    )

    now = datetime.now().replace(microsecond=0)
    submission_file_name = f"{model_name.lower()}_submission_{now}.csv"
    model_submission_df.write_csv(NIXTLA_RESULTS_PATH / submission_file_name)